# Übung 05: Fachwerk – Elementsteifigkeitsmatrizen und Lösung

Am Beispiel des Fachwerks aus der Vorlesung sollen einfache Grundfertigkeiten im Umgang mit der MFE geübt werden, insbesondere die Aufstellung von Steifigkeitsmatrizen, deren Eigenschaften im Weiteren studiert werden sollen. Des Weiteren soll der gesamte algorithmische Prozess einer FE-Berechnung von der Elementsteifigkeitsmatrix bis hin zur Berechnung der Spannungen und Verschiebungen dargestellt werden.

---

## Struktur (Abb. 1: Statisch bestimmtes Fachwerk)

Das Fachwerk besteht aus **4 Knoten** und **5 Stäben**. Die Knoten sind mit $1,\ldots,4$ nummeriert, die Stäbe mit $\boxed{1},\ldots,\boxed{5}$.

> **Lagerung:** Knoten 1 ist ein Festlager (Pin, $U_1 = U_2 = 0$), Knoten 2 ist ein Loslager (Roller, $U_4 = 0$).

---

## Tabelle 1: Geometrie- und Werkstoffdaten

| Stabnummer | $\boxed{1}$ | $\boxed{2}$ | $\boxed{3}$ | $\boxed{4}$ | $\boxed{5}$ |
|---|---|---|---|---|---|
| Elastizitätsmodul [MPa] | 210000 | 210000 | 210000 | 210000 | 210000 |
| $X_1, Y_1, X_2, Y_2\;[10^3\text{mm}]$ | 0, 0, 1, 0 | 0, 0, 1, 1 | 1, 0, 1, 1 | 1, 0, 2, 1 | 1, 1, 2, 1 |
| Querschnittsfläche [mm²] | 15 | 28.28 | 10 | 56.56 | 10 |

Die Kraft am Knoten 4 beträgt $\boldsymbol{F}^{(4)} = (0,\,-1000)^T\;\text{N}$.

---

## Freiheitsgrade

Jeder Knoten hat zwei Verschiebungs-Freiheitsgrade (DOFs):

| Knoten | DOF $x$ | DOF $y$ |
|:------:|:-------:|:-------:|
| 1      | $U_1$   | $U_2$   |
| 2      | $U_3$   | $U_4$   |
| 3      | $U_5$   | $U_6$   |
| 4      | $U_7$   | $U_8$   |

---

> **Hinweis zur Indizierung im Code**
>
> In den Aufgaben und der Vorlesung beginnt die Nummerierung bei **1** (Knoten 1–4, Stäbe 1–5).
> Im Python-Code gilt durchgehend **0-basierte Indizierung** – d.h. alle Listen und Arrays starten bei Index 0:
>
> | Bezeichnung (1-basiert) | Index im Code (0-basiert) |
> |:-----------------------:|:-------------------------:|
> | Knoten 1                | `nodal_coordinates[0]`    |
> | Knoten 2                | `nodal_coordinates[1]`    |
> | Stab $\boxed{2}$        | `elements[1]`             |
> | Stab $\boxed{4}$        | `elements[3]`             |
>
> **Faustregel:** Code-Index = Aufgaben-Nummer $- 1$

**Abb. 1: Statisch bestimmtes Fachwerk**

![Fachwerk](img/fachwerk.png)

## Imports

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Modelldaten

Koordinaten in $[\text{mm}]$, Querschnittsflächen in $[\text{mm}^2]$, $E$ in $[\text{MPa} = \text{N/mm}^2]$, Kräfte in $[\text{N}]$.

In [ ]:
# Knotenkoordinaten [X, Y] in mm  (Knoten 1..4 → Index 0..3)
nodal_coordinates = np.array([
    [   0.0,    0.0],   # Knoten 1
    [1000.0,    0.0],   # Knoten 2
    [1000.0, 1000.0],   # Knoten 3
    [2000.0, 1000.0],   # Knoten 4
])

# Elemente: [Knoten_i, Knoten_j, section_key]  (0-basiert; Stab 1..5 → Index 0..4)
elements = [
    [0, 1, "section 1"],   # Stab 1: Knoten 1–2
    [0, 2, "section 2"],   # Stab 2: Knoten 1–3
    [1, 2, "section 3"],   # Stab 3: Knoten 2–3
    [1, 3, "section 4"],   # Stab 4: Knoten 2–4
    [2, 3, "section 5"],   # Stab 5: Knoten 3–4
]

# Materialien: Youngscher Modul [MPa = N/mm²]
materials = {"steel": [210000.0]}

# Querschnitte: [A [mm²], material_key]
sections = {
    "section 1": [15.00, "steel"],
    "section 2": [28.28, "steel"],
    "section 3": [10.00, "steel"],
    "section 4": [56.56, "steel"],
    "section 5": [10.00, "steel"],
}

# Randbedingungen: [Knoten (0-basiert), Achse (0=x, 1=y), vorgegebene Verschiebung]
constraints = [
    [0, 0, 0.0],   # Knoten 1: U_1 = 0  (Festlager, x)
    [0, 1, 0.0],   # Knoten 1: U_2 = 0  (Festlager, y)
    [1, 1, 0.0],   # Knoten 2: U_4 = 0  (Loslager, y)
]

# Lasten: [Knoten (0-basiert), Achse (0=x, 1=y), Kraft [N]]
loads = [
    [3, 1, -1000.0],   # Knoten 4: F_y = -1000 N
]

---

### a) Lokale ESM des Stabes $\boxed{2}$

Geben Sie die lokale Elementsteifigkeitsmatrix (ESM) des Stabes $\boxed{2}$ numerisch an und erklären Sie, warum diese die Dimension $2 \times 2$ hat.

$$
\underline{\underline{k}}_{\text{lokal}} = \frac{EA}{L}
\begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}
\quad\longrightarrow\quad
\texttt{=(EA/L) * np.array([[1, -1], [-1, 1]])}
$$

> **Erklärung $2 \times 2$:** Im `______________` Koordinatensystem des Stabs gibt es pro Knoten genau `______` Freiheitsgrad – die Verschiebung entlang der Stabachse. Mit _______ Knoten pro Element ergibt sich eine ______$\times$______=________-dimensionale DOF-Liste und damit eine _____ $\times$ ________-ESM.

In [ ]:
# Stab 2 → Elementindex 1 (0-basiert)
e_idx = 1

i, j, sec_key = elements[e_idx]
A, mat_key    = sections[sec_key]
E             = materials[mat_key][0]
EA            = E * A

xy_e = nodal_coordinates[[i, j], :]
dx   = xy_e[1, 0] - xy_e[0, 0]
dy   = xy_e[1, 1] - xy_e[0, 1]
L    = np.sqrt(dx**2 + dy**2)

print(f"Stab 2: Knoten {i+1}–{j+1}")
print(f"  E    = {E:.0f} MPa")
print(f"  A    = {A:.2f} mm²")
print(f"  L    = {L:.4f} mm")
print(f"  EA/L = {EA/L:.4f} N/mm")

# TODO: lokale ESM (2×2)
k_lokal = 
print("\nLokale ESM k_lokal [N/mm]:")
print(k_lokal)

---

### b) Globale ESM des Stabes $\boxed{2}$

Geben Sie nun die globale ESM des Stabes $\boxed{2}$ numerisch an und erklären Sie, warum diese jetzt die Dimension $4 \times 4$ hat, obwohl es sich um ein und denselben Stab handelt.

Die Transformationsmatrix $\underline{\underline{T}}$ projiziert globale Knotenverschiebungen auf die lokale Stabachse:

$$
\underline{\underline{T}} = \begin{bmatrix}
\cos\theta & \sin\theta & 0 & 0 \\
0 & 0 & \cos\theta & \sin\theta
\end{bmatrix}
\quad\longrightarrow\quad
\texttt{T = np.array([[c, s, 0, 0], [0, 0, c, s]])}
$$

mit $c = \cos\theta = dx/L$, $\quad s = \sin\theta = dy/L$.

Die globale ESM ergibt sich durch Transformation:

$$
\underline{\underline{K}}_e = \underline{\underline{T}}^T \, \underline{\underline{k}}_{\text{lokal}} \, \underline{\underline{T}}
= \frac{EA}{L}
\begin{bmatrix}
 c^2  &  cs   & -c^2  & -cs  \\
 cs   &  s^2  & -cs   & -s^2 \\
-c^2  & -cs   &  c^2  &  cs  \\
-cs   & -s^2  &  cs   &  s^2
\end{bmatrix}
\quad\longrightarrow\quad
\texttt{Ke\_stab2 = T.T @ k\_lokal @ T}
$$

> **Erklärung $4 \times 4$:** Im globalen System hat jeder Knoten _____ Freiheitsgrade ($u_x$, $u_y$). Mit 2 Knoten ergibt sich eine (______ $\times 2$ = _______)-dimensionale DOF-Liste und damit eine $4 \times 4$-ESM. Der Stab selbst bleibt derselbe – die höhere Dimension entsteht durch die Koordinatentransformation ins globale System.

In [ ]:
c = dx / L
s = dy / L
print(f"c = cos θ = {c:.6f}")
print(f"s = sin θ = {s:.6f}")

# TODO: Transformationsmatrix T (2×4)
T = 

# TODO: globale ESM K_e des Stabes 2 (4×4)
Ke_stab2 = 

print("\nTransformationsmatrix T:")
print(T)
print("\nGlobale ESM K_e des Stabes 2 [N/mm]:")
print(Ke_stab2)

---

### c) Globale ESMen aller 5 Stäbe

Welche Beziehungen bestehen zwischen den globalen ESMen der fünf Stäbe? Geben Sie die globalen ESMen aller Stäbe unter Ausnutzung dieser Beziehungen numerisch an.

> **Beziehungen:**
> - Stäbe mit **gleichem $\theta$** haben **gleiche Struktur** der ESM (gleiche $c$, $s$).
> - Stäbe mit **gleichem $EA/L$** und gleichem $\theta$ sind **identisch**.
> - Stäbe mit gleichem $\theta$, aber unterschiedlichem $EA/L$ sind **proportional** zueinander:
>   $$\underline{\underline{K}}_e^{(a)} = \frac{(EA/L)^{(a)}}{(EA/L)^{(b)}} \cdot \underline{\underline{K}}_e^{(b)}$$
>
> Konkret: Stab $\boxed{3}$ und $\boxed{5}$ haben dieselbe Fläche $A = 10\,\text{mm}^2$, aber unterschiedliche Winkel ($90°$ vs. $0°$). Stab $\boxed{2}$ und $\boxed{4}$ sind beide diagonal ($45°$), jedoch mit unterschiedlichem $EA/L$ – ihre ESMen sind proportional ($\underline{\underline{K}}^4 = 2\,\underline{\underline{K}}^2$). Ebenso $\underline{\underline{K}}^1 = 1.5\,\underline{\underline{K}}^5$.

In [ ]:
def element_stiffness_matrix(EA, xy_e):
    """Globale Elementsteifigkeitsmatrix K_e für ein 2D-Stab-Element."""
    dx_e = xy_e[1, 0] - xy_e[0, 0]
    dy_e = xy_e[1, 1] - xy_e[0, 1]
    L_e  = np.sqrt(dx_e**2 + dy_e**2)
    c    = dx_e / L_e
    s    = dy_e / L_e

    k_lok = (EA / L_e) * np.array([[ 1., -1.],
                                    [-1.,  1.]])
    T_e = np.array([[c, s, 0., 0.],
                    [0., 0., c,  s]])
    return T_e.T @ k_lok @ T_e


# Alle globalen ESMen berechnen und ausgeben
for e, (i, j, sec_key) in enumerate(elements):
    A_e, mat_key = sections[sec_key]
    E_e          = materials[mat_key][0]
    EA_e         = E_e * A_e
    xy_e         = nodal_coordinates[[i, j], :]

    dx_e = xy_e[1, 0] - xy_e[0, 0]
    dy_e = xy_e[1, 1] - xy_e[0, 1]
    L_e  = np.sqrt(dx_e**2 + dy_e**2)
    theta_deg = np.degrees(np.arctan2(dy_e, dx_e))

    Ke = element_stiffness_matrix(EA_e, xy_e)
    print(f"Stab {e+1}  (Knoten {i+1}–{j+1},  θ = {theta_deg:.1f}°,  EA/L = {EA_e/L_e:.4f} N/mm):")
    print(Ke)
    print()

---

### d) Dimension der globalen ESM im 3D-Fall

Das Fachwerk aus Abb. 1 ist ein sog. **ebenes Fachwerk**, weil sich seine ganze Verformung nur in der $xy$-Ebene abspielen kann – oder in der Sprache der MFE: weil den Stäben der Freiheitsgrad in $z$-Richtung fehlt.

**Welche Dimension hätte die globale ESM des Stabes, falls sich das Fachwerk allgemein im Raum verformen könnte?**

> **Antwort:** Im 3D-Raum hat jeder Knoten _______ translatorische Freiheitsgrade ($u_x, u_y,$______). Ein Stab-Element mit 2 Knoten hätte also:
> $$\text{DOFs} = 2 \times ~~~~= ~~~~ \quad \Rightarrow \quad \underline{\underline{K}}_e \in \mathbb{R}^{~~~~ \times ~~~~}$$
>
> Die lokale ESM bleibt $2 \times 2$ (nur axiale Richtung), aber die Transformationsmatrix $\underline{\underline{T}} \in \mathbb{R}^{2 \times ~~~~~~}$ projiziert nun drei Komponenten pro Knoten auf die Stabachse:
> $$\underline{\underline{T}} = \begin{bmatrix} l & m & n & 0 & 0 & 0 \\ 0 & 0 & 0 & l & m & n \end{bmatrix}$$
> mit $l = \cos\alpha$, $m = \cos\beta$, $n = \cos\gamma$ (Richtungskosinus zur $x$-, $y$-, $z$-Achse).

---

### e) Koinzidenztabelle

Stellen Sie eine Koinzidenztabelle der Struktur gemäss der Nummerierung in Abb. 1 auf.

> **Koinzidenztabelle** – ordnet jedem Element $e$ die globalen DOF-Indizes $U_l$ zu, auf welche die lokalen Elementverschiebungen $u_j^e$ abgebildet werden. Im Code wird sie als kompaktes Integer-Array gespeichert und direkt zur Assemblierung genutzt.

Die Zuordnung folgt aus den Knotenindizes $i$, $j$ des Elements:

$$
\text{DOFs}^{(e)} = [2i,\; 2i+1,\; 2j,\; 2j+1]
\quad \text{(0-basiert)}
\qquad \Leftrightarrow \qquad
[2i-1,\; 2i,\; 2j-1,\; 2j]
\quad \text{(1-basiert)}
$$

Sie legt fest, an welcher Position die Einträge von $\underline{\underline{K}}^e$ in die globale SSM $\underline{\underline{K}}$ eingebaut werden:
$$\underline{\underline{K}}[\text{DOFs}^{(e)},\, \text{DOFs}^{(e)}] \mathrel{+}= \underline{\underline{K}}^e$$

In [ ]:
def incidence_table(elements):
    """Koinzidenztabelle (engl. incidence table):
    Gibt die globalen DOF-Indizes (0-basiert) je Element zurück."""
    conn = np.array([[el[0], el[1]] for el in elements], dtype=int)
    dofs = np.vstack((
        2 * conn[:, 0],
        2 * conn[:, 0] + 1,
        2 * conn[:, 1],
        2 * conn[:, 1] + 1,
    )).T
    return dofs


dof_table = incidence_table(elements)
print("\nKoinzidenztabelle (einfach):")
print(dof_table)

# --- Visualisierung Koinzidenztabelle (1-basierte Indizes) ---
lw = 28
cw = 11
ne = len(elements)

def hline():
    return "+" + "-"*lw + ("+" + "-"*cw) * ne + "+"

def row(label, values):
    cells = "".join(f"|{v:^{cw}}" for v in values)
    return f"|{label:<{lw}}{cells}|"

print("\nKoinzidenztabelle:")
print(hline())
print(row("  Elementnummer e",           [f"e = {e+1}" for e in range(ne)]))
print(hline())
print(row("  Elementverschiebung U_j^e", ["1  2  3  4"] * ne))
print(row("  wird abgebildet auf",       ["↓  ↓  ↓  ↓"] * ne))
print(row("  Gesamtverschiebung U_l",
          ["  ".join(str(d+1) for d in dof_table[e]) for e in range(ne)]))
print(hline())

---

### f) Struktursteifigkeitsmatrix (SSM) $\underline{\underline{K}}$

Welche Dimension besitzt die SSM $\underline{\underline{K}}$ und warum? Tragen Sie zuerst nur die Elemente der globalen ESM des Stabes $\boxed{2}$ in $\underline{\underline{K}}$ ein. Geben Sie dann die komplette SSM $\underline{\underline{K}}$ an.

> **Dimension:** Die Struktur hat $n_{\text{Knoten}} \times ~~~~~~ = ~~~~~~ \times ~~~~~~ = ~~~~~~$ globale DOFs.
> Die SSM ist daher $\underline{\underline{K}} \in \mathbb{R}^{~~~~~~ \times ~~~~~~}$.
>
> Die Assemblierung erfolgt durch Addition der Elementbeiträge an den durch die Koinzidenztabelle vorgegebenen Positionen:
> $$\underline{\underline{K}}[\text{DOFs}^{(e)},\, \text{DOFs}^{(e)}] \mathrel{+}= \underline{\underline{K}}^e$$

In [ ]:
ndof = 2 * len(nodal_coordinates)   # DOF 
print(f"Anzahl globaler DOFs: {ndof}  →  SSM hat Dimension {ndof}×{ndof}\n")

# --- Schritt 1: nur Stab 2 eintragen ---
K_nur_stab2 = np.zeros((ndof, ndof))  # Initialisierung
e2   = 1   # Elementindex Stab 2
i2, j2, sec2 = elements[e2]     
A2, mat2 = sections[sec2]             # Fläche und Materialnummer
E2       = materials[mat2][0]         # E-Modul
Ke2      = element_stiffness_matrix(E2 * A2, nodal_coordinates[[i2, j2], :]) #Globale Elementsteifigkeitsmatrix
edofs2   = dof_table[e2]              # Strukturfreiheitsgrade für das jeweilige Element
K_nur_stab2[np.ix_(edofs2, edofs2)] += Ke2 #WICHTIG! Assemblierung - Komponenten der Elementsteifkigkeitsmatrix werden an die richtige Stelle in der Struktursteifigkeitsmatrix mittels Einträge in der Koinzidenztabelle geschrieben.   

print("SSM nach Eintrag von Stab 2 (alle anderen Einträge = 0) [N/mm]:")
print(K_nur_stab2)

# --- Schritt 2: vollständige Assemblierung ---
def assemble_K(nodal_coordinates, elements, sections, materials):
    """Assembliert die globale SSM K."""
    dofs = incidence_table(elements)   # Koinzidenztabelle
    n    = int(np.max(dofs) + 1)
    K    = np.zeros((n, n))
    for e, (i, j, sec_key) in enumerate(elements): # Hier passiert dasselbe wie oben für Stab 2, nur als Schleife über alle Stäbe
        A_e, mat_key = sections[sec_key]
        E_e          = materials[mat_key][0]
        Ke           = element_stiffness_matrix(E_e * A_e, nodal_coordinates[[i, j], :])
        edofs        = dofs[e]
        K[np.ix_(edofs, edofs)] += Ke
    return K


K = assemble_K(nodal_coordinates, elements, sections, materials)
print("\nVollständige SSM K [N/mm]:")
print(K)
print(f"\nSymmetrie-Check  max|K − Kᵀ| = {np.max(np.abs(K - K.T)):.2e}")

---

### g) Gesamte FE-Hauptgleichung

Stellen Sie die gesamte FE-Hauptgleichung $\boldsymbol{F} = \underline{\underline{K}}\,\boldsymbol{U}$ auf. Warum kann man diese nicht nach $\boldsymbol{U}$ auflösen?

> **Antwort:** Die vollständige SSM $\underline{\underline{K}}$ ist **singulär** ($\det(\underline{\underline{K}}) = 0$), d.h. sie ist nicht invertierbar. Physikalisch bedeutet das: Ohne Lagerung ist die Struktur **nicht im Gleichgewicht** – sie kann als Starrkörper beliebig verschoben werden (Starrkörpermodus). Das Gleichungssystem besitzt unendlich viele Lösungen.
>
> Erst durch das Einarbeiten der Randbedingungen (Streichen der gesperrten Zeilen und Spalten) wird die reduzierte Steifigkeitsmatrix $\underline{\underline{K}}_{FF}$ regulär und invertierbar.

In [ ]:
print("FE-Hauptgleichung  F = K · U")
print(f"  F_full [N]    : {F_full}")
print(f"  Rang von K    : {np.linalg.matrix_rank(K)} von {ndof}")
print(f"  det(K)        : {np.linalg.det(K):.4e}")
print(f"  → K ist singulär: {np.linalg.matrix_rank(K) < ndof}")
print("  → Ohne Randbedingungen ist U nicht eindeutig bestimmbar.")

---

### h) Reduziertes Gleichungssystem und Verschiebungen $U_F$

Geben Sie die reduzierte FE-Hauptgleichung $\boldsymbol{F}_F = \underline{\underline{K}}_{FF}\,\boldsymbol{U}_F$ an.

**Tipp:** Alle Lagerbedingungen sind homogen, d.h. $U_U = 0$, sodass man einfach die entsprechenden Zeilen und Spalten von $\underline{\underline{K}}$ bzw. Zeilen von $\boldsymbol{F}$ und $\boldsymbol{U}$ streichen kann, um das reduzierte LGS zu erhalten.

Berechnen Sie dann die Verschiebungen $\boldsymbol{U}_F$:

$$
\boldsymbol{U}_F = \underline{\underline{K}}_{FF}^{-1}\,\boldsymbol{F}_F
\quad\longrightarrow\quad
\texttt{U\_F = np.linalg.solve(K\_FF, F\_F)}
$$

In [ ]:
# Schritt 1: Es werden Masken gebaut, die angeben, welche Verschiebungsfreiheitsgrade bekannt sind (mask_U) und welche Werte sie annehmen (U_U). Ebenso für die äusseren Lasten (mask_F, F_F)
mask_U = np.zeros(ndof, dtype=bool) 
U_U_vals = []
for node, axis, val in constraints:
    dof = 2 * int(node) + int(axis)
    mask_U[dof] = True
    U_U_vals.append(val)

U_U    = np.array(U_U_vals, dtype=float)
mask_F = ~mask_U   # freie DOFs

print("Freie DOFs (F-Partition):",    [i+1 for i, v in enumerate(mask_F) if v])
print("Gesperrte DOFs (U-Partition):", [i+1 for i, v in enumerate(mask_U) if v])

# Partitionierung
K_FF = K[np.ix_(mask_F, mask_F)]
K_FU = K[np.ix_(mask_F, mask_U)]
# Globaler Lastvektor F_full [N]
F_full = np.zeros(ndof)
for node, axis, force in loads:
    F_full[2 * int(node) + int(axis)] = force
F_F  = F_full[mask_F]

print("\nReduzierte Steifigkeitsmatrix K_FF [N/mm]:")
print(K_FF)
print("\nReduzierter Lastvektor F_F [N]:", F_F)

# TODO: freie Verschiebungen lösen  U_F = K_FF^{-1} (F_F - K_FU @ U_U)
U_F = 

# Gesamtverschiebungsvektor aufbauen
U = np.zeros(ndof)
U[mask_U] = U_U
U[mask_F] = U_F

print("\nVerschiebungen U_F [mm] (freie DOFs):")
free_labels = [i+1 for i, v in enumerate(mask_F) if v]
for label, val in zip(free_labels, U_F):
    print(f"  U_{label} = {val:+.6e} mm")

print("\nGesamter Verschiebungsvektor U [mm]:")
for k in range(ndof):
    print(f"  U_{k+1} = {U[k]:+.6e} mm")

---

### i) Reaktionskräfte $\boldsymbol{F}_U$

Berechnen Sie die Reaktionskräfte $\boldsymbol{F}_U$:

$$
\boldsymbol{F}_U = \underline{\underline{K}}_{UF}\,\boldsymbol{U}_F
\quad\longrightarrow\quad
\texttt{F\_U = K\_UF @ U\_F}
$$

In [ ]:
K_UF = K[np.ix_(mask_U, mask_F)]
K_UU = K[np.ix_(mask_U, mask_U)]

# TODO: Reaktionskräfte F_U
F_U = 

F_full[mask_U] = F_U

print("Reaktionskräfte F_U [N] (gesperrte DOFs):")
fixed_labels = [i+1 for i, v in enumerate(mask_U) if v]
for label, val in zip(fixed_labels, F_U):
    print(f"  F_{label} = {val:+.4f} N")

print("\nGleichgewichts-Check:")
print(f"  ΣF_x = {np.sum(F_full[0::2]):+.6f} N  (soll 0 sein)")
print(f"  ΣF_y = {np.sum(F_full[1::2]):+.6f} N  (soll 0 sein)")

---

### j) Spannung im Stab $\boxed{2}$

Berechnen Sie die Spannung im Stab $\boxed{2}$.

**Nachlaufrechnung** (Post-Processing):

1. Globale Elementverschiebungen extrahieren: $U^{(e)} = [u_{x,i},\, u_{y,i},\, u_{x,j},\, u_{y,j}]^T$
2. Auf lokale Achse projizieren: $u_{\text{lokal}} = T\,U^{(e)}$
3. Dehnung und Spannung (Hooke):

$$
\varepsilon_e = \frac{u_{\text{lokal},j} - u_{\text{lokal},i}}{L}
\quad\longrightarrow\quad
\texttt{eps2 = (u\_lok2[1] - u\_lok2[0]) / L2}
$$

$$
\sigma_e = E\,\varepsilon_e
\quad\longrightarrow\quad
\texttt{sigma2 = E2 * eps2}
$$

> **Vorzeichen-Konvention:** $\sigma_e > 0$ → Zug, $\sigma_e < 0$ → Druck.

In [ ]:
# --- Spannung in Stab 2 ---
e_idx2 = 1   # Stab 2 → Index 1
i2, j2, sec2 = elements[e_idx2] #Knotennummer und Querschnittsnummern des Stabes 2
A2, mat2 = sections[sec2] 
E2       = materials[mat2][0]
xy2      = nodal_coordinates[[i2, j2], :]

dx2 = xy2[1, 0] - xy2[0, 0]
dy2 = xy2[1, 1] - xy2[0, 1]
L2  = np.sqrt(dx2**2 + dy2**2)
c2  = dx2 / L2
s2  = dy2 / L2

T2 = np.array([[c2, s2, 0., 0.],
               [0., 0., c2, s2]])

# Globale Elementverschiebungen
U_e2 = U[[2*i2, 2*i2+1, 2*j2, 2*j2+1]]

# Lokale Verschiebungen
u_lok2 = T2 @ U_e2

# TODO: Dehnung ε (lokale Verlängerung / Länge)
eps2 = 

# TODO: Spannung σ [MPa]
sigma2 = E2*eps2

print(f"Stab 2  (Knoten {i2+1}–{j2+1}):")
print(f"  Lokale Verschiebungen u_lokal = {u_lok2} mm")
print(f"  Dehnung        ε = {eps2:+.6e}")
print(f"  Spannung       σ = {sigma2:+.4f} MPa  "
      f"({'Zug' if sigma2 > 0 else 'Druck'})")

# --- Übersicht: alle Stäbe ---
print("\n" + "─" * 70)
print(f"{'Stab':>5}  {'Knoten':>8}  {'L [mm]':>8}  {'θ [°]':>7}  "
      f"{'ε [-]':>13}  {'σ [MPa]':>10}  Zustand")
print("─" * 70)
for e, (i, j, sec_key) in enumerate(elements):
    A_e, mat_key = sections[sec_key]
    E_e          = materials[mat_key][0]
    xy_e         = nodal_coordinates[[i, j], :]
    dx_e = xy_e[1,0]-xy_e[0,0];  dy_e = xy_e[1,1]-xy_e[0,1]
    L_e  = np.sqrt(dx_e**2+dy_e**2)
    c_e  = dx_e/L_e;  s_e = dy_e/L_e
    T_e  = np.array([[c_e, s_e, 0., 0.], [0., 0., c_e, s_e]])
    U_e  = U[[2*i, 2*i+1, 2*j, 2*j+1]]
    u_l  = T_e @ U_e
    eps  = (u_l[1] - u_l[0]) / L_e
    sig  = E_e * eps
    theta_d = np.degrees(np.arctan2(dy_e, dx_e))
    z    = "Zug" if sig > 0 else ("Druck" if sig < 0 else "0")
    print(f"  {e+1:>3}  {i+1:>3}–{j+1:<3}  "
          f"  {L_e:>6.1f}  {theta_d:>7.1f}  "
          f"  {eps:>13.6e}  {sig:>10.4f}  {z}")